# 01 · 数据准备：清洗 + 5-core + Leave-last-out

**数据**：[Amazon Reviews 2023](https://amazon-reviews-2023.github.io/)（McAuley Lab）· Movies_and_TV 原始评论 + 商品元数据

**输入**：`Movies_and_TV.jsonl.gz`、`meta_Movies_and_TV.jsonl.gz`。
如果 `DRIVE_DIR` 里没有这两个文件，会自动从官方地址下载。

**输出**（`OUT_DIR`）：
- `train/valid/test_data.npz`：双塔直接使用的 `user` / `hist` / `item` 数组
- `id_maps.npz`：编号 ↔ 原始 ID 对照表 + 训练集商品频率
- `interactions.parquet`：清洗后的完整交互表（带 `split` 列）

在 Colab 上用 CPU 运行时即可，完整运行约 30 分钟。

In [ ]:
# ---------------- 1. 路径 & 配置 ----------------
import os, gzip, json, time, shutil, urllib.request
import numpy as np
import pandas as pd

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive"            # 两个 .gz 文件所在的 Drive 文件夹，按实际位置改
    OUT_DIR   = "/content/drive/MyDrive/two_tower_movies_tv"
    LOCAL_DIR = "/content/amazon2023"               # Colab 本地磁盘，读得比 Drive 快
else:
    DRIVE_DIR = OUT_DIR = LOCAL_DIR = os.environ.get("DATA_DIR", "./data/amazon2023")

CFG = dict(
    category = "Movies_and_TV",
    min_rating = None,   # None = 所有评论都当正样本；设成 4 则只保留 rating >= 4
    k_core = 5,          # 用户、商品都至少 5 条交互（与官方 5-core benchmark 一致）
    max_hist = 20,       # 用户塔使用的最近历史长度
    chunk = 1_000_000,   # 解析 jsonl 时每块行数
)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(LOCAL_DIR, exist_ok=True)

REVIEWS_GZ = f"{CFG['category']}.jsonl.gz"
META_GZ    = f"meta_{CFG['category']}.jsonl.gz"

BASE_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"
URLS = {REVIEWS_GZ: f"{BASE_URL}/review_categories/{REVIEWS_GZ}",
        META_GZ:    f"{BASE_URL}/meta_categories/{META_GZ}"}

def download(url, path):
    t0, tmp = time.time(), path + ".part"
    def hook(blocks, bsize, total):
        if total > 0 and blocks % 5000 == 0:
            print(f"  {blocks*bsize/1e9:.2f} / {total/1e9:.2f} GB", end="\r")
    urllib.request.urlretrieve(url, tmp, hook)
    os.replace(tmp, path)
    print(f"\n下载完成 {os.path.basename(path)}: {os.path.getsize(path)/1e9:.2f} GB, {time.time()-t0:.0f}s")

# 优先用 DRIVE_DIR 里已有的文件（复制到本地磁盘读得更快）；没有就从官方地址下载
for f in (REVIEWS_GZ, META_GZ):
    src, dst = os.path.join(DRIVE_DIR, f), os.path.join(LOCAL_DIR, f)
    if os.path.exists(dst):
        continue
    if os.path.exists(src):
        t0 = time.time(); shutil.copy(src, dst)
        print(f"复制 {f}: {os.path.getsize(dst)/1e9:.2f} GB, {time.time()-t0:.0f}s")
    else:
        print(f"{src} 不存在，从官方地址下载 {URLS[f]}")
        download(URLS[f], dst)

In [ ]:
# ---------------- 2. 流式解析评论，只保留 4 列 ----------------
# 整个文件有上千万条评论、带全文；一次性 pd.read_json 会爆内存，所以按块读、只取需要的字段。
# 如果之前已经解析过，直接读缓存的 parquet。
RAW_PARQUET = os.path.join(OUT_DIR, "reviews_raw_4cols.parquet")

def parse_reviews(path, chunk):
    parts, rows, t0 = [], [], time.time()
    def flush():
        df = pd.DataFrame(rows, columns=["user_id", "item_id", "rating", "timestamp"])
        df["user_id"] = df["user_id"].astype("category")
        df["item_id"] = df["item_id"].astype("category")
        df["rating"] = df["rating"].astype("float32")
        df["timestamp"] = df["timestamp"].astype("int64")
        parts.append(df); rows.clear()
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for n, line in enumerate(f, 1):
            r = json.loads(line)
            rows.append((r["user_id"], r["parent_asin"], r["rating"], r["timestamp"]))
            if len(rows) >= chunk:
                flush(); print(f"  已解析 {n:,} 行, {time.time()-t0:.0f}s", end="\r")
    if rows:
        flush()
    # 合并各块的 categorical（各块类别不同，需 union）
    from pandas.api.types import union_categoricals
    out = pd.DataFrame({
        "user_id": union_categoricals([p["user_id"] for p in parts]),
        "item_id": union_categoricals([p["item_id"] for p in parts]),
        "rating": np.concatenate([p["rating"].to_numpy() for p in parts]),
        "timestamp": np.concatenate([p["timestamp"].to_numpy() for p in parts]),
    })
    print(f"\n解析完成 {len(out):,} 条，用时 {time.time()-t0:.0f}s")
    return out

if os.path.exists(RAW_PARQUET):
    reviews = pd.read_parquet(RAW_PARQUET)
    print(f"读取缓存 {len(reviews):,} 条")
else:
    reviews = parse_reviews(os.path.join(LOCAL_DIR, REVIEWS_GZ), CFG["chunk"])
    reviews.to_parquet(RAW_PARQUET, index=False)   # 以后不用再解析 2.4 GB 的 gz

In [ ]:
# ---------------- 3. 清洗：去重 + (可选) 只保留高分 + k-core ----------------
df = reviews
if CFG["min_rating"] is not None:
    df = df[df["rating"] >= CFG["min_rating"]]

# 同一用户对同一商品多次评论：保留时间最晚的一次
df = df.sort_values(["user_id", "timestamp"], kind="stable")
df = df.drop_duplicates(["user_id", "item_id"], keep="last")

# 迭代 k-core：删掉交互太少的用户/商品后，其它人的计数也会变，所以要循环到稳定
def k_core(df, k):
    it = 0
    while True:
        it += 1
        # observed=True / 去掉 0：categorical 的 value_counts 会包含已删除的类别（计数 0），否则会死循环
        uc = df["user_id"].value_counts(); uc = uc[uc > 0]
        ic = df["item_id"].value_counts(); ic = ic[ic > 0]
        bad_u = uc[uc < k].index
        bad_i = ic[ic < k].index
        print(f"  第 {it} 轮: {len(df):,} 条, 用户 {len(uc):,}, 商品 {len(ic):,}")
        if len(bad_u) == 0 and len(bad_i) == 0:
            return df
        df = df[~df["user_id"].isin(bad_u) & ~df["item_id"].isin(bad_i)]

if CFG["k_core"]:
    df = k_core(df, CFG["k_core"])

# 删掉没用到的类别，重新整数编码（0 留作 padding）
df = df.copy()
df["user_id"] = df["user_id"].cat.remove_unused_categories()
df["item_id"] = df["item_id"].cat.remove_unused_categories()
df["u"] = df["user_id"].cat.codes.astype("int32") + 1
df["i"] = df["item_id"].cat.codes.astype("int32") + 1
user_ids = np.asarray(df["user_id"].cat.categories)   # u-1 -> 原始 user_id
item_ids = np.asarray(df["item_id"].cat.categories)   # i-1 -> 原始 parent_asin
N_USERS, N_ITEMS = len(user_ids), len(item_ids)
print(f"清洗后：用户 {N_USERS:,} | 商品 {N_ITEMS:,} | 交互 {len(df):,}")

In [ ]:
# ---------------- 4. Leave-last-out 划分 ----------------
# 关键：按 (用户, 时间) 排序；时间相同的保留原始文件顺序（stable），保证可复现。
df = df.sort_values(["u", "timestamp"], kind="stable").reset_index(drop=True)

# 每个用户从后往前数的位置：0 = 最后一条，1 = 倒数第二条
rank_from_end = df.groupby("u", sort=False).cumcount(ascending=False).to_numpy()
df["split"] = np.where(rank_from_end == 0, "test",
              np.where(rank_from_end == 1, "valid", "train"))

# 检查：每个用户恰好 1 条 test、1 条 valid，且 train 时间 <= valid 时间 <= test 时间
cnt = df.groupby(["u", "split"], observed=True).size().unstack(fill_value=0)
assert (cnt["test"] == 1).all() and (cnt["valid"] == 1).all(), "每个用户应有 1 条 test 和 1 条 valid"
t = df.pivot_table(index="u", columns="split", values="timestamp", aggfunc="max")
assert (t["train"] <= t["valid"]).all() and (t["valid"] <= t["test"]).all(), "时间顺序有问题"
print(df["split"].value_counts())

In [ ]:
# ---------------- 5. 无穿越的历史序列 ----------------
# 对每条交互，取同一用户在它【之前】的最近 K 个商品（第 0 列 = 最近一次）。
# - train 样本：只看到更早的 train 交互
# - valid 样本：看到全部 train 交互
# - test  样本：看到 train + valid 交互
u_arr, i_arr = df["u"].to_numpy(), df["i"].to_numpy()
n, K = len(df), CFG["max_hist"]
hist = np.zeros((n, K), dtype=np.int32)
for k in range(1, K + 1):
    same_user = np.zeros(n, dtype=bool)
    same_user[k:] = u_arr[k:] == u_arr[:-k]
    prev = np.zeros(n, dtype=np.int32)
    prev[k:] = i_arr[:-k]
    hist[:, k - 1] = np.where(same_user, prev, 0)

split = df["split"].to_numpy()
data = {s: {"user": u_arr[split == s], "hist": hist[split == s], "item": i_arr[split == s]}
        for s in ("train", "valid", "test")}
for s, d in data.items():
    print(f"{s:5s}: {len(d['user']):,}")

# 冷启动商品：在 valid/test 出现、但 train 里从没出现过（ID embedding 学不到，只能靠元数据特征）
train_items = np.zeros(N_ITEMS + 1, dtype=bool); train_items[data["train"]["item"]] = True
for s in ("valid", "test"):
    cold = (~train_items[data[s]["item"]]).mean()
    print(f"{s} 中冷启动商品占比: {cold:.2%}")

# logQ 纠偏用的商品频率，只能用 train 统计
item_freq = np.bincount(data["train"]["item"], minlength=N_ITEMS + 1).astype("float64")

In [ ]:
# ---------------- 6. 保存，后面训练直接读 ----------------
df[["user_id", "item_id", "u", "i", "rating", "timestamp", "split"]].to_parquet(
    os.path.join(OUT_DIR, "interactions.parquet"), index=False)
for s, d in data.items():
    np.savez_compressed(os.path.join(OUT_DIR, f"{s}_data.npz"), **d)
np.savez_compressed(os.path.join(OUT_DIR, "id_maps.npz"),
                    user_ids=user_ids, item_ids=item_ids, item_freq=item_freq)
print("已保存到", OUT_DIR)

In [ ]:
# ---------------- 之后在训练 notebook 里这样读 ----------------
# d = {s: dict(np.load(f"{OUT_DIR}/{s}_data.npz")) for s in ("train", "valid", "test")}
# train_data, valid_data, test_data = d["train"], d["valid"], d["test"]
# maps = np.load(f"{OUT_DIR}/id_maps.npz", allow_pickle=True)
# user_ids, item_ids, item_freq = maps["user_ids"], maps["item_ids"], maps["item_freq"]
# N_USERS, N_ITEMS = len(user_ids), len(item_ids)